In [40]:
import pandas as pd
import gc

df = pd.read_csv("../data/fines.csv")

def method1(df):
    res = []
    for i in range(len(df)):
        fine = df.iloc[i]['Fines']
        refund = df.iloc[i]['Refund']
        year = df.iloc[i]['Year']
        calculated = fine/(year*refund)
        res.append(calculated)
    return res

def method2(df):
    res = []
    for index, row in df.iterrows():
        calculated = row['Fines']/(row['Refund']*row['Year'])
        res.append(calculated) 
    return res

def method3(df):
    return df.apply(lambda row: row['Fines']/(row['Refund']*row['Year']), axis=1)

def method4(df):
    return df['Fines']/(df['Refund']*df['Year'])

def method5(df):
    return df['Fines'].values / df['Refund'].values * df['Year'].values


In [2]:
%%timeit
df['calculated_1'] = method1(df)

54.9 ms ± 1.34 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
%%timeit
df['calculated_2'] = method2(df)

20.1 ms ± 464 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
%%timeit
df['calculated_3'] = method3(df)

6.53 ms ± 206 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
%%timeit
df['calculated_4'] = method4(df)

111 μs ± 944 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [14]:
%%timeit
df['calculated_5'] = method5(df)

50.7 μs ± 920 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [18]:
%timeit row = df[df['CarNumber'] == 'O136HO197RUS']
df_indexed = df.set_index('CarNumber')
%timeit row = df_indexed.loc['O136HO197RUS']

203 μs ± 15.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
35.5 μs ± 581 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [19]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CarNumber     930 non-null    object 
 1   Refund        930 non-null    int64  
 2   Fines         930 non-null    float64
 3   Make          930 non-null    object 
 4   Model         917 non-null    object 
 5   Year          930 non-null    int64  
 6   calculated_4  930 non-null    float64
 7   calculated_5  930 non-null    float64
 8   calculated_3  930 non-null    float64
 9   calculated_2  930 non-null    float64
dtypes: float64(5), int64(2), object(3)
memory usage: 203.9 KB


In [21]:
optimized_df = df.copy()
float_cols = optimized_df.select_dtypes(include=['float64']).columns
for col in float_cols:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast='float')

int_cols = optimized_df.select_dtypes(include=['int64']).columns
for col in int_cols:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast='integer')

optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CarNumber     930 non-null    object 
 1   Refund        930 non-null    int8   
 2   Fines         930 non-null    float32
 3   Make          930 non-null    object 
 4   Model         917 non-null    object 
 5   Year          930 non-null    int16  
 6   calculated_4  930 non-null    float32
 7   calculated_5  930 non-null    float64
 8   calculated_3  930 non-null    float32
 9   calculated_2  930 non-null    float32
dtypes: float32(4), float64(1), int16(1), int8(1), object(3)
memory usage: 177.6 KB


In [28]:
object_cols = optimized_df.select_dtypes(include=['object']).columns
for col in object_cols:
    optimized_df[col] = optimized_df[col].astype('category')
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   CarNumber     930 non-null    category
 1   Refund        930 non-null    int8    
 2   Fines         930 non-null    float32 
 3   Make          930 non-null    category
 4   Model         917 non-null    category
 5   Year          930 non-null    int16   
 6   calculated_4  930 non-null    float32 
 7   calculated_5  930 non-null    float64 
 8   calculated_3  930 non-null    float32 
 9   calculated_2  930 non-null    float32 
dtypes: category(3), float32(4), float64(1), int16(1), int8(1)
memory usage: 78.4 KB


In [41]:
del df
gc.collect()

1344